In [5]:
!pip -q install opencv-python numpy matplotlib scikit-image imageio imageio-ffmpeg

In [6]:
import shutil
import subprocess
from pathlib import Path

import cv2
import imageio
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

ROOT = Path('/content/Problem3')
ROOT.mkdir(exist_ok=True)
VIDEO_PATH = ROOT / 'campus_video.mp4'
FRAMES_DIR = ROOT / 'frames'
COLMAP_DIR = ROOT / 'colmap'
SCENE_DIR = ROOT / 'scene'
GS_DIR = ROOT / 'gaussian_splatting'
GS_OUTPUT = ROOT / 'gs_output'
RENDERS_DIR = ROOT / 'renders'
OUTPUT_VIDEO = ROOT / 'gt_vs_3dgs.mp4'

TARGET_FRAMES = 300
TRAIN_ITERATIONS = 30000

REPORT = {
    'training_iterations': TRAIN_ITERATIONS,
    'num_gaussians': None,
    'psnr_train': None,
    'ssim_train': None,
    'video_path_or_url': str(OUTPUT_VIDEO),
    'failure_modes': (
        'Blur on thin structures, floaters in free space, exposure drift between views, '
        'pose drift if COLMAP fails on textureless regions.'
    ),
}

if not VIDEO_PATH.exists():
    from google.colab import drive
    drive.mount('/content/drive')
    mydrive = Path('/content/drive/MyDrive')
    roots = [mydrive] + [p for p in mydrive.iterdir() if p.is_dir()]
    sid = Path('/content/drive/.shortcut-targets-by-id/1fUo6TQmvkCPWg9MVaEQTyx3tOZ0yTEkG')
    if sid.exists():
        roots.append(sid)
    names = ('campus_video.mp4', 'campus.mp4')
    found = next((mydrive / n for n in names if (mydrive / n).exists()), None)
    if not found:
        found = next((p for n in names for r in roots for p in r.rglob(n)), None)
    if not found:
        seen = [str(p.relative_to(mydrive)) for p in mydrive.rglob('*.mp4')][:10]
        raise FileNotFoundError(f'Video not found. MP4 files in My Drive: {seen or "none"}')
    shutil.copy(found, VIDEO_PATH)

print(f'Using {VIDEO_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using /content/Problem3/campus_video.mp4


In [7]:
def get_video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()
    return fps


def extract_frames(video_path, out_dir, target_frames=300):
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    step = max(total // target_frames, 1)
    fps_out = get_video_fps(video_path) / step
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-vf', f'select=not(mod(n\\,{step})),setpts=N/FRAME_RATE/TB',
        '-vsync', 'vfr', str(out_dir / 'frame_%05d.jpg'),
    ]
    subprocess.run(cmd, check=True)
    n = len(list(out_dir.glob('*.jpg')))
    return n, fps_out


if len(list(FRAMES_DIR.glob('*.jpg'))) < 10:
    n_frames, INPUT_FPS = extract_frames(VIDEO_PATH, FRAMES_DIR, TARGET_FRAMES)
else:
    n_frames = len(list(FRAMES_DIR.glob('*.jpg')))
    INPUT_FPS = get_video_fps(VIDEO_PATH)

print(f'{n_frames} frames in {FRAMES_DIR}, fps≈{INPUT_FPS:.2f}')

363 frames in /content/Problem3/frames, fps≈10.00


## COLMAP — GT poses and intrinsics K

Run on GPU/project node if Colab lacks COLMAP. Set `RUN_COLMAP = True` when ready.

In [8]:
RUN_COLMAP = False  # set True on compute node with colmap installed

COLMAP_DIR.mkdir(exist_ok=True)
db = COLMAP_DIR / 'database.db'
sparse = COLMAP_DIR / 'sparse'

if RUN_COLMAP:
    subprocess.run(['colmap', 'feature_extractor', '--database_path', str(db),
                      '--image_path', str(FRAMES_DIR)], check=True)
    subprocess.run(['colmap', 'exhaustive_matcher', '--database_path', str(db)], check=True)
    sparse.mkdir(exist_ok=True)
    subprocess.run(['colmap', 'mapper', '--database_path', str(db),
                      '--image_path', str(FRAMES_DIR), '--output_path', str(sparse)], check=True)
    print('COLMAP done:', list(sparse.rglob('*'))[:5])
else:
    print('Skip COLMAP. Run on project compute, then copy colmap/sparse/0 here.')

Skip COLMAP. Run on project compute, then copy colmap/sparse/0 here.


In [9]:
def prepare_3dgs_scene(frames_dir, colmap_sparse0, scene_dir):
    images = scene_dir / 'images'
    dst_sparse = scene_dir / 'sparse' / '0'
    images.mkdir(parents=True, exist_ok=True)
    dst_sparse.mkdir(parents=True, exist_ok=True)
    for src in sorted(frames_dir.glob('*.jpg')):
        link = images / src.name
        if not link.exists():
            shutil.copy(src, link)
    for f in ['cameras.bin', 'images.bin', 'points3D.bin']:
        src = colmap_sparse0 / f
        if src.exists():
            shutil.copy(src, dst_sparse / f)
    return scene_dir

colmap_sparse0 = COLMAP_DIR / 'sparse' / '0'
if colmap_sparse0.exists():
    prepare_3dgs_scene(FRAMES_DIR, colmap_sparse0, SCENE_DIR)
    print(f'3DGS scene ready: {SCENE_DIR}')
else:
    print('Waiting for COLMAP sparse/0 output')

Waiting for COLMAP sparse/0 output


## 3D Gaussian Splatting

Official repo: https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/

Clone on project GPU node, then set `RUN_TRAIN = True`.

In [10]:
RUN_TRAIN = False  # set True on GPU node with 3DGS env

if RUN_TRAIN:
    if not GS_DIR.exists():
        subprocess.run([
            'git', 'clone', 'https://github.com/graphdeco-inria/gaussian-splatting.git',
            str(GS_DIR),
        ], check=True)
    subprocess.run([
        'python', str(GS_DIR / 'train.py'),
        '-s', str(SCENE_DIR), '-m', str(GS_OUTPUT),
        '--iterations', str(TRAIN_ITERATIONS),
    ], check=True)
    print('Training done')
else:
    print(f'''
On project GPU:
  git clone https://github.com/graphdeco-inria/gaussian-splatting.git {GS_DIR}
  python {GS_DIR}/train.py -s {SCENE_DIR} -m {GS_OUTPUT} --iterations {TRAIN_ITERATIONS}
  python {GS_DIR}/render.py -m {GS_OUTPUT}
Copy renders to {RENDERS_DIR}
''')


On project GPU:
  git clone https://github.com/graphdeco-inria/gaussian-splatting.git /content/Problem3/gaussian_splatting
  python /content/Problem3/gaussian_splatting/train.py -s /content/Problem3/scene -m /content/Problem3/gs_output --iterations 30000
  python /content/Problem3/gaussian_splatting/render.py -m /content/Problem3/gs_output
Copy renders to /content/Problem3/renders



In [11]:
def read_num_gaussians(gs_output):
    ply = gs_output / 'point_cloud' / f'iteration_{TRAIN_ITERATIONS}' / 'point_cloud.ply'
    if not ply.exists():
        ply = next(gs_output.rglob('point_cloud.ply'), None)
    if ply is None:
        return None
    with open(ply) as f:
        for line in f:
            if line.startswith('element vertex'):
                return int(line.split()[-1])
    return None

REPORT['num_gaussians'] = read_num_gaussians(GS_OUTPUT)
REPORT['num_gaussians']

In [12]:
def eval_train_metrics(frames_dir, renders_dir):
    gt_files = sorted(frames_dir.glob('*.jpg'))
    psnrs, ssims = [], []
    for gt_path in gt_files:
        rd_path = renders_dir / gt_path.name
        if not rd_path.exists():
            rd_path = renders_dir / gt_path.stem.replace('frame_', '') / gt_path.name
        if not rd_path.exists():
            continue
        gt = cv2.cvtColor(cv2.imread(str(gt_path)), cv2.COLOR_BGR2RGB)
        rd = cv2.cvtColor(cv2.imread(str(rd_path)), cv2.COLOR_BGR2RGB)
        if rd.shape != gt.shape:
            rd = cv2.resize(rd, (gt.shape[1], gt.shape[0]))
        psnrs.append(psnr(gt, rd, data_range=255))
        ssims.append(ssim(gt, rd, channel_axis=2, data_range=255))
    return float(np.mean(psnrs)), float(np.mean(ssims))


if RENDERS_DIR.exists() and any(RENDERS_DIR.rglob('*.jpg')):
    REPORT['psnr_train'], REPORT['ssim_train'] = eval_train_metrics(FRAMES_DIR, RENDERS_DIR)
    print(f"PSNR: {REPORT['psnr_train']:.3f}, SSIM: {REPORT['ssim_train']:.4f}")
else:
    print(f'Put 3DGS renders in {RENDERS_DIR} (same filenames as frames/)')

Put 3DGS renders in /content/Problem3/renders (same filenames as frames/)


In [13]:
def make_side_by_side(frames_dir, renders_dir, out_path, fps):
    gt_files = sorted(frames_dir.glob('*.jpg'))
    writer = imageio.get_writer(str(out_path), fps=fps)
    for gt_path in gt_files:
        rd_path = renders_dir / gt_path.name
        if not rd_path.exists():
            continue
        gt = cv2.imread(str(gt_path))
        rd = cv2.imread(str(rd_path))
        if rd.shape != gt.shape:
            rd = cv2.resize(rd, (gt.shape[1], gt.shape[0]))
        combo = np.hstack([gt, rd])
        writer.append_data(cv2.cvtColor(combo, cv2.COLOR_BGR2RGB))
    writer.close()


if RENDERS_DIR.exists() and any(RENDERS_DIR.rglob('*.jpg')):
    make_side_by_side(FRAMES_DIR, RENDERS_DIR, OUTPUT_VIDEO, INPUT_FPS)
    print(f'Saved {OUTPUT_VIDEO}')
else:
    print('Side-by-side video will be written after renders are available')

Side-by-side video will be written after renders are available


In [14]:
REPORT

{'training_iterations': 30000,
 'num_gaussians': None,
 'psnr_train': None,
 'ssim_train': None,
 'video_path_or_url': '/content/Problem3/gt_vs_3dgs.mp4',
 'failure_modes': 'Blur on thin structures, floaters in free space, exposure drift between views, pose drift if COLMAP fails on textureless regions.'}